# H19a: Does Partial Singular-Value Equalization Create Saddle Points in the Loss Landscape?

---

## Motivation and Scientific Context

In experiment **H3b**, we observed a striking and counterintuitive result: *partial* SV equalization (equalizing only the top-k singular values, with k < full rank) performed **worse than plain SGD** — not merely weaker, but *actively destructive* to optimization. This demands explanation.

### The Hypothesis

Partial SV equalization creates a **geometrically inconsistent** update direction. When we equalize only the top-k singular values of a layer's gradient while leaving the remaining singular values untouched, we produce a hybrid object:
- The **equalized subspace** (top-k SVs) wants to explore uniformly in all k directions — it has been "flattened" into an isotropic update within that subspace.
- The **non-equalized subspace** (remaining SVs) retains the original anisotropic gradient structure, pointing preferentially along directions of large curvature.

This mismatch at the boundary between equalized and non-equalized subspaces may introduce **negative Hessian eigenvalues** — i.e., the partial-SV step actively moves the parameters *toward saddle points* in the loss landscape.

### Why This Matters for the Muon-as-RG-Gauge-Fixing Theory

If confirmed, this result explains why Muon's effectiveness requires **full** SV equalization (the polar factor, where all SVs become 1): any partial equalization breaks the gauge symmetry inconsistently, creating geometric frustration. The polar factor is the unique point where the update direction is purely rotational — no singular-value anisotropy at all — and thus geometrically self-consistent.

### Experimental Design

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| Architecture | 2-layer deep linear, 4x4 | 32 total parameters — small enough for exact Hessian computation |
| Warmup | 50 SGD+momentum steps | Reach a non-trivial point in parameter space |
| k values | {0, 1, 2, 3, 4} per layer | 0 = SGD (no equalization), 4 = full polar factor, 1-3 = partial |
| Hessian method | Finite differences | Exact for 32x32 Hessian, no approximation needed |
| Seeds | 5 | Statistical robustness across random initializations |

### Key Prediction

The number of negative Hessian eigenvalues (saddle directions) should **peak at intermediate k** — neither at k=0 (SGD, which follows the natural gradient landscape) nor at k=4 (full Muon/polar factor, which is geometrically consistent), but at k=1, 2, or 3 where the geometric inconsistency is maximal.

In [ ]:
import numpy as np
import os

print("=" * 90)
print("H19a: PARTIAL SV EQUALIZATION — SADDLE POINT ANALYSIS")
print("=" * 90)
print(f"NumPy version: {np.__version__}")
print("Dependencies loaded: numpy (linear algebra, SVD, eigendecomposition), os (paths)")

In [ ]:
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
print(f"Script directory: {SCRIPT_DIR}")

## Experimental Constants

We use a deliberately tiny network (2-layer, 4x4 = 32 parameters) so the full 32x32 Hessian matrix can be computed exactly via finite differences. This is crucial: approximate Hessian methods (Gauss-Newton, diagonal, etc.) could miss the very saddle-point structure we are looking for. The finite-difference step size `FD_EPS=1e-5` balances numerical precision against floating-point noise.

In [ ]:
DIM = 4
N_LAYERS = 2
N_PARAMS = N_LAYERS * DIM * DIM  # 32
WARMUP_STEPS = 50
NUM_SEEDS = 5
MOMENTUM = 0.9
LR_WARMUP = 0.01
LR_STEP = 0.01
FD_EPS = 1e-5
NS_ITERS = 5

print("--- Experimental Configuration ---")
print(f"  Network:       {N_LAYERS}-layer deep linear, {DIM}x{DIM} per layer")
print(f"  Total params:  {N_PARAMS} (full Hessian is {N_PARAMS}x{N_PARAMS} = {N_PARAMS**2} entries)")
print(f"  Warmup:        {WARMUP_STEPS} SGD+momentum steps (momentum={MOMENTUM}, LR={LR_WARMUP})")
print(f"  Test step LR:  {LR_STEP}")
print(f"  FD epsilon:    {FD_EPS} (for Hessian finite differences)")
print(f"  NS iterations: {NS_ITERS} (Newton-Schulz, for polar factor reference)")
print(f"  Seeds:         {NUM_SEEDS} independent random seeds")

In [ ]:
# For a 4x4 matrix, SVD has 4 SVs. We test k per-layer from 0..4.
# But we also want to test higher k values conceptually by thinking of
# the "flattened" gradient. To keep it concrete: k is per-layer.
# k=0: SGD direction (no equalization)
# k=1..3: partial equalization
# k=4: full equalization (polar factor)
K_PER_LAYER = [0, 1, 2, 3, 4]

print("--- k Values (per-layer SV equalization depth) ---")
print(f"  k values tested: {K_PER_LAYER}")
print(f"  k=0:   Pure SGD direction (SVs scaled to match polar norm, but ratios preserved)")
print(f"  k=1:   Only the largest SV is 'equalized' (trivially, to itself) — minimal modification")
print(f"  k=2:   Top-2 SVs set to their mean — beginning of geometric conflict")
print(f"  k=3:   Top-3 SVs equalized, 1 untouched — strong partial equalization")
print(f"  k={DIM}:   Full polar factor (all SVs -> 1) — complete equalization = Muon")
print(f"  Total Hessian computations per seed: {len(K_PER_LAYER)} (one per k value)")
print(f"  Each Hessian requires {2 * N_PARAMS} gradient evaluations (central differences)")

---

## Network Architecture and Core Functions

The network is a **2-layer deep linear network**: $f(X) = W_2 \cdot W_1 \cdot X$. Despite being linear (no activations), deep linear networks are a rich testbed for optimization geometry because:

1. **The loss landscape has saddle points** — even for linear networks, the parameterization through products of matrices creates non-convexity.
2. **SVD structure is directly relevant** — each weight matrix $W_l$ has a well-defined SVD, and the gradient $\nabla_{W_l} \mathcal{L}$ inherits spectral structure from the interaction between $W_l$ and the data covariance.
3. **Full Hessian is tractable** — with only 32 parameters, we can compute the exact $32 \times 32$ Hessian via finite differences.

### Helper Functions

- `pack/unpack`: Convert between a flat parameter vector $\theta \in \mathbb{R}^{32}$ and the list of layer weight matrices $[W_1, W_2]$.
- `forward`: Compute the network output $W_2 W_1 X$.
- `loss_fn`: Mean squared error $\frac{1}{2N} \sum_i \|f(x_i) - y_i\|^2$.
- `grad_fn`: Backpropagation through the linear layers, returning both packed and per-layer gradients.
- `hessian_fd`: Full Hessian via central finite differences — the gold standard for small parameter counts.

In [ ]:
def pack(Ws):
    """Flatten list of weight matrices into a single parameter vector."""
    return np.concatenate([W.ravel() for W in Ws])

print(f"[pack] Concatenates {N_LAYERS} weight matrices ({DIM}x{DIM} each) into theta of length {N_PARAMS}")

In [ ]:
def unpack(theta):
    """Reshape flat parameter vector back into list of weight matrices."""
    Ws = []
    idx = 0
    for _ in range(N_LAYERS):
        Ws.append(theta[idx:idx + DIM * DIM].reshape(DIM, DIM))
        idx += DIM * DIM
    return Ws

print(f"[unpack] Splits theta[0:{N_PARAMS}] -> W1[{DIM}x{DIM}], W2[{DIM}x{DIM}]")

In [ ]:
def forward(Ws, X):
    """Forward pass: out = W_{N_LAYERS} ... W_1 X (deep linear)."""
    out = X.copy()
    for W in Ws:
        out = W @ out
    return out

print(f"[forward] Computes W2 @ W1 @ X — effective linear map is W_eff = W2 * W1")

In [ ]:
def loss_fn(theta, X, Y):
    """MSE loss: L = (1/2N) sum_i ||f(x_i) - y_i||^2."""
    Ws = unpack(theta)
    pred = forward(Ws, X)
    return 0.5 * np.mean(np.sum((pred - Y) ** 2, axis=0))

print("[loss_fn] L(theta) = 0.5 * mean_over_samples( ||pred - target||^2 )")

In [ ]:
def grad_fn(theta, X, Y):
    """
    Backpropagation for the deep linear network.
    
    Returns:
        packed_grad: flat gradient vector (length N_PARAMS)
        grads: list of per-layer gradient matrices [dL/dW1, dL/dW2]
        
    The per-layer gradients are crucial because partial SV equalization
    operates on each layer's gradient matrix independently.
    """
    Ws = unpack(theta)
    N = X.shape[1]
    acts = [X.copy()]
    for W in Ws:
        acts.append(W @ acts[-1])
    delta = (acts[-1] - Y) / N
    grads = []
    for l in range(N_LAYERS - 1, -1, -1):
        grads.insert(0, delta @ acts[l].T)
        if l > 0:
            delta = Ws[l].T @ delta
    return pack(grads), grads

print("[grad_fn] Standard backprop through linear layers")
print("  Returns both packed gradient (for Hessian FD) and per-layer gradients (for SV equalization)")

In [ ]:
def hessian_fd(theta, X, Y):
    """
    Full Hessian via central finite differences.
    
    For each parameter i, we compute:
        H[:, i] = (grad(theta + eps*e_i) - grad(theta - eps*e_i)) / (2*eps)
    
    Then symmetrize: H = 0.5*(H + H^T) to eliminate numerical asymmetry.
    
    This is the EXACT Hessian (up to FD truncation error O(eps^2)).
    For 32 parameters, this requires 64 gradient evaluations — fast enough.
    """
    n = len(theta)
    H = np.zeros((n, n))
    for i in range(n):
        tp = theta.copy()
        tp[i] += FD_EPS
        tm = theta.copy()
        tm[i] -= FD_EPS
        gp, _ = grad_fn(tp, X, Y)
        gm, _ = grad_fn(tm, X, Y)
        H[:, i] = (gp - gm) / (2 * FD_EPS)
    return 0.5 * (H + H.T)

print(f"[hessian_fd] Central finite differences with eps={FD_EPS}")
print(f"  Computes full {N_PARAMS}x{N_PARAMS} Hessian — {2*N_PARAMS} gradient evaluations per call")
print(f"  Symmetrization ensures H = H^T (removes O(eps^3) asymmetry artifacts)")

---

## Partial Singular-Value Equalization: The Core Manipulation

This is the heart of the experiment. Given a layer gradient matrix $G_l \in \mathbb{R}^{4 \times 4}$ with SVD $G_l = U \Sigma V^T$, partial SV equalization modifies the singular values as follows:

### How partial equalization works for each k:

- **k = 0 (SGD baseline):** Keep original SV ratios, but rescale to match the Frobenius norm of the polar factor ($\sqrt{d}$). This ensures any differences we see are due to *direction*, not step size.
- **k = 1:** Set $\sigma_1' = \sigma_1$ (trivially "equalize" only the top SV to itself). This is effectively the same as k=0 after norm matching.
- **k = 2:** Set $\sigma_1' = \sigma_2' = \frac{\sigma_1 + \sigma_2}{2}$, then rescale. Now the top-2 SVs are flattened but $\sigma_3, \sigma_4$ retain their original ratios — **geometric conflict begins**.
- **k = 3:** Set $\sigma_1' = \sigma_2' = \sigma_3' = \text{mean}(\sigma_1, \sigma_2, \sigma_3)$, only $\sigma_4$ is free.
- **k = 4 (full Muon):** All SVs become equal: $\sigma_i' = 1$ for all $i$. This is the polar factor $UV^T$.

### The norm-matching detail

All variants are rescaled to have Frobenius norm $\sqrt{d} = 2.0$, matching the polar factor. This is critical: without norm matching, differences in negative eigenvalue counts could be an artifact of different step magnitudes rather than directional geometry.

In [ ]:
def partial_sv_equalize_layer(M, k):
    """
    Equalize top-k SVs of a layer gradient matrix M (DIM x DIM).
    k=0: return M (no equalization, just norm-matched)
    k=DIM: full polar factor (UV^T)
    k=1..DIM-1: partial equalization
    """
    U, sigma, Vt = np.linalg.svd(M, full_matrices=False)
    d = len(sigma)

    if k == 0:
        # No equalization, but match norm to polar factor for fair comparison
        target_norm = np.sqrt(d)  # ||UV^T||_F = sqrt(d)
        current_norm = np.linalg.norm(sigma)
        if current_norm > 1e-15:
            sigma_scaled = sigma * (target_norm / current_norm)
            return U @ np.diag(sigma_scaled) @ Vt
        return M

    kk = min(k, d)
    if kk >= d:
        # Full polar factor
        return U @ Vt

    # Equalize top-k SVs to their mean
    sigma_new = sigma.copy()
    top_mean = np.mean(sigma[:kk])
    sigma_new[:kk] = top_mean

    # Scale to match polar factor norm
    target_norm = np.sqrt(d)
    current_norm = np.linalg.norm(sigma_new)
    if current_norm > 1e-15:
        sigma_new *= target_norm / current_norm

    return U @ np.diag(sigma_new) @ Vt

print("[partial_sv_equalize_layer] Defined.")
print(f"  Input: {DIM}x{DIM} gradient matrix M with SVD M = U diag(sigma) V^T")
print(f"  k=0: norm-match only (preserve SV ratios, scale to ||polar||_F = sqrt({DIM}) = {np.sqrt(DIM):.4f})")
print(f"  k=1..{DIM-1}: equalize top-k SVs to their mean, then norm-match")
print(f"  k={DIM}: full polar factor U @ V^T (all SVs become 1)")

In [ ]:
def newton_schulz_layer(M, n_iters=NS_ITERS):
    """
    Newton-Schulz iteration for polar factor approximation.
    
    Not used in the main experiment (we use exact SVD-based equalization),
    but included as a reference implementation of how Muon computes
    the polar factor in practice.
    """
    norm = np.linalg.norm(M, 'fro')
    if norm < 1e-15:
        return M
    X = M / norm
    for _ in range(n_iters):
        A = X.T @ X
        X = 1.5 * X - 0.5 * X @ A
    return X

print(f"[newton_schulz_layer] {NS_ITERS}-iteration Newton-Schulz polar factor approximation")
print("  (Reference only — main experiment uses exact SVD-based equalization for precision)")

---

## Main Experiment: Probing the Hessian After Partial-SV Steps

### Protocol for each seed:

1. **Initialize** the network near identity: $W_l = I + 0.1 \cdot \mathcal{N}(0,1)$, with random data $X, Y \sim 0.3 \cdot \mathcal{N}(0,1)$.
2. **Warm up** for 50 SGD+momentum steps to reach a non-trivial point in parameter space (away from initialization but not yet converged).
3. **Snapshot** the parameter vector $\theta_{\text{warm}}$ and compute the full Hessian $H(\theta_{\text{warm}})$ as a baseline. Count negative eigenvalues (saddle directions) before any partial-SV intervention.
4. **For each k in {0, 1, 2, 3, 4}:**
   - Compute the gradient at $\theta_{\text{warm}}$.
   - Apply partial-SV equalization with depth $k$ to each layer's gradient.
   - Take one step: $\theta_k = \theta_{\text{warm}} - \eta \cdot g_{\text{equalized}}$.
   - Compute the full Hessian $H(\theta_k)$ at the new point.
   - Count: negative eigenvalues (saddle directions), near-zero eigenvalues (flat directions), minimum eigenvalue, and loss.

### What we are looking for:

- **Intermediate k produces MORE negative eigenvalues** than either k=0 (SGD) or k=4 (full Muon).
- The minimum eigenvalue is MORE negative for intermediate k.
- This would confirm that partial equalization *creates* saddle-point geometry that neither pure SGD nor full Muon produces.

In [ ]:
def main():
    seeds = [42 + i * 137 for i in range(NUM_SEEDS)]

    print("=" * 100)
    print("H19a: PARTIAL SV EQUALIZATION CREATES SADDLE POINTS?")
    print("=" * 100)
    print(f"Network: {N_LAYERS}-layer deep linear {DIM}x{DIM} ({N_PARAMS} params)")
    print(f"k per layer: {K_PER_LAYER}")
    print(f"Warmup: {WARMUP_STEPS} SGD steps at LR={LR_WARMUP}, then one step at LR={LR_STEP}")
    print(f"Seeds: {NUM_SEEDS}")
    print(f"Random seeds: {seeds}")
    print()

    # Accumulators
    neg_eigs_before_all = []
    neg_eigs_after = {k: [] for k in K_PER_LAYER}
    min_eig_after = {k: [] for k in K_PER_LAYER}
    near_zero_after = {k: [] for k in K_PER_LAYER}
    loss_after = {k: [] for k in K_PER_LAYER}

    for si, seed in enumerate(seeds):
        rng = np.random.RandomState(seed)
        X = rng.randn(DIM, 64) * 0.3
        Y = rng.randn(DIM, 64) * 0.3
        weights = [np.eye(DIM) + rng.randn(DIM, DIM) * 0.1 for _ in range(N_LAYERS)]
        theta = pack(weights)

        print(f"\n{'─' * 100}")
        print(f"  SEED {si + 1}/{NUM_SEEDS} (seed={seed})")
        print(f"{'─' * 100}")
        print(f"  Data: X shape={X.shape}, Y shape={Y.shape}, scale=0.3")
        print(f"  Init: W_l = I + 0.1*N(0,1), initial ||theta|| = {np.linalg.norm(theta):.4f}")

        # Print initial SV structure of each weight matrix
        for l, W in enumerate(weights):
            svs = np.linalg.svd(W, compute_uv=False)
            print(f"  W{l+1} initial SVs: [{', '.join(f'{s:.4f}' for s in svs)}]  "
                  f"(condition number: {svs[0]/svs[-1]:.2f})")

        initial_loss = loss_fn(theta, X, Y)
        print(f"  Initial loss: {initial_loss:.6e}")

        # Warmup with SGD + momentum
        mom = np.zeros_like(theta)
        for step in range(WARMUP_STEPS):
            g, _ = grad_fn(theta, X, Y)
            mom = MOMENTUM * mom + g
            theta = theta - LR_WARMUP * mom

        warmup_loss = loss_fn(theta, X, Y)
        print(f"\n  After {WARMUP_STEPS}-step SGD warmup:")
        print(f"    Loss: {initial_loss:.6e} -> {warmup_loss:.6e}  "
              f"(reduction: {initial_loss/max(warmup_loss,1e-30):.1f}x)")
        print(f"    ||theta_warm|| = {np.linalg.norm(theta):.4f}")

        # Print warmup SV structure
        ws_warm = unpack(theta)
        for l, W in enumerate(ws_warm):
            svs = np.linalg.svd(W, compute_uv=False)
            print(f"    W{l+1} warmed SVs: [{', '.join(f'{s:.4f}' for s in svs)}]  "
                  f"(condition: {svs[0]/svs[-1]:.2f})")

        # Hessian BEFORE the partial-SV step
        print(f"\n  Computing {N_PARAMS}x{N_PARAMS} Hessian at theta_warm...")
        H_before = hessian_fd(theta, X, Y)
        eigs_before = np.linalg.eigvalsh(H_before)
        n_neg_before = int(np.sum(eigs_before < -1e-8))
        n_near_zero_before = int(np.sum(np.abs(eigs_before) < 1e-6))
        loss_before = loss_fn(theta, X, Y)
        neg_eigs_before_all.append(n_neg_before)

        print(f"  Hessian BEFORE partial-SV step:")
        print(f"    Negative eigenvalues (< -1e-8):  {n_neg_before}")
        print(f"    Near-zero eigenvalues (|e| < 1e-6): {n_near_zero_before}")
        print(f"    Eigenvalue range: [{eigs_before[0]:.4e}, {eigs_before[-1]:.4e}]")
        print(f"    Loss at this point: {loss_before:.6e}")
        print(f"    Top-5 eigenvalues:  [{', '.join(f'{e:.4e}' for e in eigs_before[-5:])}]")
        print(f"    Bottom-5 eigenvalues: [{', '.join(f'{e:.4e}' for e in eigs_before[:5])}]")

        # Compute gradient SVD info at the warmup point (for interpreting k effects)
        _, grads_at_warm = grad_fn(theta, X, Y)
        print(f"\n  Gradient SVD structure at theta_warm (determines what k=1,2,3 actually do):")
        for l, gl in enumerate(grads_at_warm):
            svs_g = np.linalg.svd(gl, compute_uv=False)
            print(f"    grad(W{l+1}) SVs: [{', '.join(f'{s:.4e}' for s in svs_g)}]  "
                  f"ratio sigma1/sigma4 = {svs_g[0]/max(svs_g[-1],1e-15):.1f}")

        # For each k: take one partial-SV step, measure Hessian
        print(f"\n  --- Sweeping k values: measuring Hessian AFTER one partial-SV step ---")
        for k in K_PER_LAYER:
            theta_k = theta.copy()
            g, grads_list = grad_fn(theta_k, X, Y)

            # Apply partial SV equalization per layer
            step_layers = []
            for l in range(N_LAYERS):
                step_layers.append(partial_sv_equalize_layer(grads_list[l], k))
            step_vec = pack(step_layers)

            # Report the SV structure of the equalized step
            step_norm = np.linalg.norm(step_vec)
            cos_with_grad = np.dot(step_vec, g) / (np.linalg.norm(step_vec) * np.linalg.norm(g) + 1e-30)

            theta_new = theta_k - LR_STEP * step_vec

            # Hessian after step
            H_after = hessian_fd(theta_new, X, Y)
            eigs_after_k = np.linalg.eigvalsh(H_after)
            n_neg = int(np.sum(eigs_after_k < -1e-8))
            min_eig = eigs_after_k[0]
            n_near_zero = int(np.sum(np.abs(eigs_after_k) < 1e-6))
            l_after = loss_fn(theta_new, X, Y)

            neg_eigs_after[k].append(n_neg)
            min_eig_after[k].append(min_eig)
            near_zero_after[k].append(n_near_zero)
            loss_after[k].append(l_after)

            # Per-layer equalized SV info
            layer_sv_info = []
            for l in range(N_LAYERS):
                svs_eq = np.linalg.svd(step_layers[l], compute_uv=False)
                layer_sv_info.append(f"[{', '.join(f'{s:.3f}' for s in svs_eq)}]")

            label = {0: "SGD (norm-matched)", 1: "top-1 equalized", 2: "top-2 equalized",
                     3: "top-3 equalized", DIM: "FULL polar factor"}
            print(f"\n    k={k} ({label.get(k, f'top-{k} equalized')}):")
            print(f"      Step ||g_eq|| = {step_norm:.4f}, cos(g_eq, g_raw) = {cos_with_grad:.4f}")
            for l in range(N_LAYERS):
                print(f"      Layer {l+1} equalized SVs: {layer_sv_info[l]}")
            print(f"      After step: {n_neg} neg eigs, {n_near_zero} near-zero, "
                  f"min_eig={min_eig:.4e}")
            print(f"      Loss: {loss_before:.6e} -> {l_after:.6e}  "
                  f"(delta={l_after - loss_before:+.4e})")

    # =========================================================================
    # RESULTS
    # =========================================================================
    print(f"\n\n{'=' * 100}")
    print("RESULTS: NEGATIVE HESSIAN EIGENVALUES AFTER ONE PARTIAL-SV STEP")
    print(f"{'=' * 100}")

    print(f"\n  {'k':>4}  {'Mean neg eigs':>14}  {'Std neg eigs':>14}  "
          f"{'Mean min eig':>14}  {'Mean near-0':>12}  {'Mean loss':>14}")
    print("  " + "-" * 80)
    for k in K_PER_LAYER:
        print(f"  {k:>4}  {np.mean(neg_eigs_after[k]):>14.1f}  "
              f"{np.std(neg_eigs_after[k]):>14.1f}  "
              f"{np.mean(min_eig_after[k]):>14.4e}  "
              f"{np.mean(near_zero_after[k]):>12.1f}  "
              f"{np.mean(loss_after[k]):>14.6e}")

    # KEY TEST: Does intermediate k produce MORE negative eigenvalues?
    print(f"\n  === KEY TEST: Negative eigenvalue count peaks at intermediate k ===")
    mean_neg_0 = np.mean(neg_eigs_after[0])
    mean_neg_full = np.mean(neg_eigs_after[DIM])
    endpoints_max = max(mean_neg_0, mean_neg_full)

    peak_k = None
    peak_neg = -1
    for k in K_PER_LAYER:
        mn = np.mean(neg_eigs_after[k])
        if mn > peak_neg:
            peak_neg = mn
            peak_k = k

    intermediate_ks = [k for k in K_PER_LAYER if 0 < k < DIM]
    intermediate_worse = any(
        np.mean(neg_eigs_after[k]) > endpoints_max + 0.5
        for k in intermediate_ks
    )

    print(f"    k=0 (SGD) mean neg eigs:       {mean_neg_0:.1f}")
    print(f"    k={DIM} (full Muon) mean neg eigs: {mean_neg_full:.1f}")
    for k in intermediate_ks:
        mn = np.mean(neg_eigs_after[k])
        delta = mn - endpoints_max
        print(f"    k={k} (partial) mean neg eigs:  {mn:.1f}  (delta vs endpoints max: {delta:+.1f})")
    print(f"    Peak at k={peak_k} with {peak_neg:.1f} neg eigs")
    print(f"    Intermediate k MORE saddle dirs: {'CONFIRMED' if intermediate_worse else 'NOT CONFIRMED'}")

    # Interpretation of the key test
    print(f"\n  --- Interpretation ---")
    if intermediate_worse:
        print("    RESULT: Partial SV equalization DOES create additional saddle directions.")
        print("    The geometric inconsistency between equalized and non-equalized subspaces")
        print("    introduces negative curvature along their mixing directions. This explains")
        print("    why partial equalization (H3b) was WORSE than SGD: not just a weaker update,")
        print("    but one that actively moves toward saddle points.")
    else:
        print("    RESULT: Partial SV equalization does NOT clearly create more saddle directions")
        print("    than the endpoints. The hypothesis about geometric inconsistency causing")
        print("    saddle points may need refinement. Possible explanations:")
        print("    - The effect may be too small to detect with 5 seeds at 4x4 scale")
        print("    - Saddle creation may require more steps (not just one)")
        print("    - The harm from partial equalization (H3b) may arise from a different mechanism")

    # Secondary analysis: loss degradation
    print(f"\n  === Loss comparison ===")
    for k in K_PER_LAYER:
        ml = np.mean(loss_after[k])
        ml0 = np.mean(loss_after[0])
        ratio = ml / max(ml0, 1e-30)
        status = "WORSE" if ratio > 1.05 else ("BETTER" if ratio < 0.95 else "SIMILAR")
        print(f"    k={k}: loss={ml:.6e}  (ratio vs k=0: {ratio:.4f}x)  [{status}]")

    # Hessian spectrum shape analysis
    print(f"\n  === Mean Hessian eigenvalue spectrum (before step) ===")
    print(f"    Mean neg eigs before step: {np.mean(neg_eigs_before_all):.1f}")
    print(f"    This tells us the landscape geometry at the warmup point:")
    if np.mean(neg_eigs_before_all) > 0:
        print(f"    -> The warmup point already has saddle structure ({np.mean(neg_eigs_before_all):.1f} neg eigs on average)")
        print(f"    -> Any ADDITIONAL neg eigs from partial SV steps are truly CREATED by the equalization")
    else:
        print(f"    -> The warmup point is at/near a local minimum (no neg eigs)")
        print(f"    -> Any neg eigs after partial SV steps demonstrate saddle CREATION")

    # Min eigenvalue analysis (complementary to count)
    print(f"\n  === Minimum eigenvalue analysis (how DEEP are the saddle directions?) ===")
    for k in K_PER_LAYER:
        mean_min = np.mean(min_eig_after[k])
        std_min = np.std(min_eig_after[k])
        print(f"    k={k}: mean min_eig = {mean_min:.4e} +/- {std_min:.4e}")

    print(f"\n{'=' * 100}")
    print("EXPERIMENT COMPLETE")
    print(f"{'=' * 100}")

### Execute the Experiment

Running the main loop: 5 seeds x 5 k-values = 25 Hessian computations (plus 5 baseline Hessians = 30 total). Each Hessian requires 64 gradient evaluations. Total: ~1920 gradient evaluations.

In [ ]:
if __name__ == '__main__':
    main()

---

## Conclusions and Implications

### What This Experiment Tests

This experiment directly probes whether the *mechanism* behind partial SV equalization's poor performance (observed in H3b) is the creation of saddle-point geometry. By computing the **exact** Hessian eigenspectrum after a single partial-SV step, we can distinguish between:

1. **Saddle creation** (negative Hessian eigenvalues appear or increase) -- the partial-SV step moves parameters into regions of negative curvature.
2. **Slow convergence** (eigenvalues stay positive but become small) -- the step is simply inefficient.
3. **Directional misalignment** (loss increases without saddle creation) -- the step points in the wrong direction entirely.

### Implications for the Muon-as-RG-Gauge-Fixing Theory

- **If confirmed:** Partial SV equalization creates saddle points because it breaks the gauge symmetry *inconsistently*. The polar factor (full Muon) is special precisely because it is the **unique** norm-preserving update direction that does not create geometric conflict between equalized and non-equalized subspaces -- there IS no non-equalized subspace.
- **If not confirmed:** The harm from partial equalization may operate through a subtler mechanism, such as accumulated trajectory effects over many steps rather than single-step Hessian changes. This would point toward **compounding directional errors** (see H20a) rather than instantaneous saddle creation.

### Connection to Other Experiments

| Experiment | Relationship |
|------------|-------------|
| **H3b** | Established that partial equalization is WORSE than SGD -- this experiment explains WHY |
| **H19b** | Tests whether the Hessian eigenvectors align with the SV basis (complementary geometric analysis) |
| **H20a** | Tests cosine compounding over many steps (alternative explanation if saddle hypothesis fails) |
| **H15** | Normalization-gauge duality -- partial equalization may break this duality |